In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# --- Cell 1: Cài đặt môi trường ---
!pip install streamlit pyngrok pyarrow pandas numpy torch matplotlib seaborn tqdm openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 81.5 MB/s eta 0:00:00


In [3]:
# --- Cell 3: Viết file dashboard.py (Phiên bản cuối cùng - Đã xóa Font) ---
%%writefile dashboard.py
import streamlit as st
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# XÓA: Phần cài đặt font đã được lược bỏ

# --- FILE PATHS ---
FRESH_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/datastorm/50k/model/"
ONLINE_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/datastorm/online-retail/model/"
HOLIDAY_PATH = "/content/drive/MyDrive/Colab Notebooks/datastorm/holiday/"
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/datastorm/50k/data/"

# --- CACHING: Tải model và dữ liệu chỉ một lần ---
@st.cache_resource
def load_model_and_scaler():
    class ImputationNet(nn.Module):
        def __init__(self, input_dim=52, output_dim=24, hidden=256):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, output_dim))
        def forward(self, x): return self.net(x)

    model = ImputationNet()
    model.load_state_dict(torch.load(os.path.join(FRESH_MODEL_PATH, "latent_demand_model_final.pth"), map_location='cpu'))
    model.eval()

    with open(os.path.join(FRESH_MODEL_PATH, "scaler_X_final.pkl"), "rb") as f:
        scaler_X = pickle.load(f)
    return model, scaler_X

@st.cache_data
def load_all_data():
    df_source = pd.read_parquet(os.path.join(DATA_PATH, "train.parquet"))

    if len(df_source) > 1_000_000:
        df_source = df_source.sample(n=20_000, random_state=42).reset_index(drop=True)

    df_source['dt'] = pd.to_datetime(df_source['dt'])

    holiday_df = pd.read_csv(os.path.join(HOLIDAY_PATH, "vietnam_holidays_2024_2025.csv"))
    holiday_df['date'] = pd.to_datetime(holiday_df['date'])

    wholesale_pred = pd.read_csv(os.path.join(ONLINE_MODEL_PATH, "wholesale_next_order_predictions.csv"))
    return df_source, holiday_df, wholesale_pred

# --- PIPELINE XỬ LÝ DỮ LIỆU ---
# Hàm này không cần cache vì nó phụ thuộc vào df_source đã được lấy mẫu
def run_full_pipeline(_df_source, _holiday_df, _wholesale_pred, _model, _scaler):
    df = _df_source.copy()

    def prepare_data(row):
        sales, stock = np.array(row['hours_sale']), np.array(row['hours_stock_status'])
        return np.concatenate([sales * stock, stock, [row['discount'], row['precpt'], row['holiday_flag'], row['avg_temperature']]])

    X_source = np.array([prepare_data(row) for _, row in df.iterrows()])
    X_source_scaled = _scaler.transform(X_source)
    with torch.no_grad():
        Y_pred = _model(torch.tensor(X_source_scaled, dtype=torch.float32)).numpy()

    df['daily_latent_demand'] = np.maximum(Y_pred[:, 6:23].sum(axis=1), 0)

    df['holiday_name'] = df['dt'].map(_holiday_df.set_index('date')['holiday_name']).fillna('None')
    holiday_multipliers = {"Tết Nguyên Đán": 2.3, "Tết Trung Thu": 1.8, "Ngày Giải phóng Miền Nam": 1.3, "Quốc tế Lao động": 1.3, "Quốc khánh": 1.3, "Khai giảng": 1.5}
    df['adjusted_demand'] = df.apply(lambda row: row['daily_latent_demand'] * holiday_multipliers.get(row['holiday_name'], 1.0), axis=1)

    normalized_wholesale_demand = 0.5
    df['final_demand'] = df['adjusted_demand'] + normalized_wholesale_demand

    return df

# --- HÀM VẼ BIỂU ĐỒ ---
def plot_dashboard(df_plot, high_thresh, low_thresh):
    daily_summary = df_plot.groupby('dt').agg(
        final_demand=('final_demand', 'mean'),
        avg_discount=('discount', 'mean'),
        avg_stockout_hours=('stock_hour6_22_cnt', 'mean')
    ).reset_index()

    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(18, 10), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

    ax1.bar(daily_summary['dt'], daily_summary['final_demand'], label='Dự báo Demand Tổng hợp', zorder=5, color='skyblue', alpha=0.7, width=0.5)
    ax1.axhline(high_thresh, color='red', linestyle='--', label=f'Ngưỡng cao ({high_thresh:.2f})')
    ax1.axhline(low_thresh, color='orange', linestyle='--', label=f'Ngưỡng thấp ({low_thresh:.2f})')

    action_map = {'📉 Giảm giá 10%': ('green', 'Giảm giá', 'v'), '🔥 Tăng nhập gấp 150%': ('red', 'Tăng nhập gấp', 'o'), '📈 Tăng tồn 20%': ('blue', 'Tăng tồn', 's')}
    for action_str, (color, label, marker) in action_map.items():
        subset = df_plot[df_plot['action'] == action_str]
        if not subset.empty:
            ax1.scatter(subset['dt'], subset['final_demand'], color=color, s=100, zorder=15, label=label, edgecolors='white', marker=marker)

    holiday_dates = df_plot[df_plot['holiday_name'] != 'None']['dt'].unique()
    if len(holiday_dates) > 0:
        ax1.axvspan(min(holiday_dates) - pd.Timedelta(hours=12), max(holiday_dates) + pd.Timedelta(hours=12), color='yellow', alpha=0.2, label='Kỳ nghỉ lễ')

    ax1.set_ylabel('Demand (chuẩn hóa)')
    ax1.grid(True, linestyle='--', alpha=0.7)

    ax2.plot(daily_summary['dt'], daily_summary['avg_discount'], color='purple', linestyle=':', marker='^', label='Discount trung bình')
    ax2.set_ylabel('Discount', color='purple')
    ax2.tick_params(axis='y', labelcolor='purple')
    ax2.invert_yaxis()

    ax2b = ax2.twinx()
    ax2b.bar(daily_summary['dt'], daily_summary['avg_stockout_hours'], color='gray', alpha=0.5, width=0.4, label='Số giờ hết hàng TB')
    ax2b.set_ylabel('Số giờ hết hàng', color='gray')
    ax2b.tick_params(axis='y', labelcolor='gray')

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    handles2b, labels2b = ax2b.get_legend_handles_labels()
    ax1.legend(handles1 + handles2 + handles2b, labels1 + labels2 + labels2b, loc='upper left')

    st.pyplot(fig)

# --- GIAO DIỆN STREAMLIT ---
def main():
    st.set_page_config(layout="wide")
    st.title("🚀 RetailDemand Copilot: Dashboard Dự Báo & Đề Xuất Hành Động")

    # Load model và dữ liệu thô
    model, scaler_X = load_model_and_scaler()
    df_source_raw, holiday_df, wholesale_pred = load_all_data()

    # Logic lấy mẫu được thực hiện bên trong hàm load_all_data
    df_source = df_source_raw

    # Chạy pipeline xử lý dữ liệu
    with st.spinner(f"Đang xử lý {len(df_source)} dòng dữ liệu và chạy mô hình..."):
        df_processed = run_full_pipeline(df_source, holiday_df, wholesale_pred, model, scaler_X)
        st.success("Xử lý thành công!")

    st.sidebar.header("Bộ lọc Tùy chỉnh")

    product_list = sorted(df_processed['product_id'].unique())
    selected_product = st.sidebar.selectbox("Chọn Product ID để phân tích:", product_list, index=product_list.index(4) if 4 in product_list else 0)

    min_date, max_date = df_processed['dt'].min().date(), df_processed['dt'].max().date()
    default_start = max_date - pd.Timedelta(days=6)
    selected_start, selected_end = st.sidebar.date_input(
        "Chọn khoảng thời gian:",
        value=[default_start, max_date], min_value=min_date, max_value=max_date
    )

    df_filtered = df_processed[
        (df_processed['product_id'] == selected_product) &
        (df_processed['dt'].dt.date >= selected_start) &
        (df_processed['dt'].dt.date <= selected_end)
    ].copy()

    low_thresh, high_thresh = np.quantile(df_processed['final_demand'], [0.25, 0.75])
    def get_action(row, h, l):
        if row['final_demand'] > h: return "🔥 Tăng nhập gấp 150%" if row['stock_hour6_22_cnt'] > 3 else "📈 Tăng tồn 20%"
        elif row['final_demand'] < l: return "📉 Giảm giá 10%"
        else: return "✅ Duy trì"
    df_filtered['action'] = df_filtered.apply(lambda row: get_action(row, high_thresh, low_thresh), axis=1)

    if df_filtered.empty:
        st.warning("Không tìm thấy dữ liệu cho sản phẩm này trong khoảng thời gian đã chọn.")
    else:
        st.header(f"Phân tích cho Product ID: {selected_product}")

        total_demand = df_filtered['final_demand'].sum()
        days_to_restock = len(df_filtered[df_filtered['action'] == '🔥 Tăng nhập gấp 150%'])
        avg_stockout_rate = df_filtered['stock_hour6_22_cnt'].mean() / 17 * 100

        col1, col2, col3 = st.columns(3)
        col1.metric("Tổng Demand Dự báo", f"{total_demand:.2f}")
        col2.metric("Số ngày cần Tăng nhập gấp", f"{days_to_restock}")
        col3.metric("Tỷ lệ Stockout trung bình", f"{avg_stockout_rate:.1f}%")

        plot_dashboard(df_filtered, high_thresh, low_thresh)

        st.subheader("Bảng Kế Hoạch Hành Động")
        st.dataframe(df_filtered[df_filtered['action'] != '✅ Duy trì'][['dt', 'store_id', 'final_demand', 'action', 'stock_hour6_22_cnt', 'discount']])

        with st.expander("Xem Dữ liệu chi tiết"):
            st.dataframe(df_filtered)

if __name__ == '__main__':
    main()

Writing dashboard.py


In [4]:
# CELL 3: Thiết lập Ngrok và Chạy Streamlit (PHIÊN BẢN CHỐNG LỖI)

from pyngrok import ngrok
import time
import os

# --- Bước 1: Kiểm tra xem file dashboard.py có tồn tại không ---
if not os.path.exists('dashboard.py'):
    print("LỖI: Không tìm thấy file 'dashboard.py'. Vui lòng chạy lại Cell 2 để tạo file này trước.")
else:
    # --- Bước 2: Thiết lập Ngrok ---
    # DÁN AUTHTOKEN CỦA BẠN VÀO ĐÂY
    NGROK_AUTH_TOKEN = "1vVronbJNeILnRWL0DMEDassGu7_5pFWSA1MKzMKgoVhLwHue"

    if NGROK_AUTH_TOKEN == "DÁN_AUTHTOKEN_CỦA_BẠN_VÀO_ĐÂY":
        print("LỖI: Vui lòng dán ngrok authtoken của bạn vào biến NGROK_AUTH_TOKEN.")
    else:
        # Ngắt các tunnel cũ đang chạy để tránh xung đột
        ngrok.kill()

        # Thiết lập authtoken
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)

        # --- Bước 3: Chạy Streamlit trong nền ---
        # Lệnh `nohup` và `&` đảm bảo nó chạy ổn định trong background
        print("Đang khởi động ứng dụng Streamlit trong nền...")
        get_ipython().system_raw('nohup streamlit run dashboard.py &')

        # --- Bước 4 (QUAN TRỌNG): Thêm khoảng chờ ---
        # Cho Streamlit 5 giây để khởi động, tải mô hình và mở cổng 8501
        print("Đang chờ ứng dụng khởi động (5 giây)...")
        time.sleep(5)

        # --- Bước 5: Kết nối Ngrok và hiển thị URL ---
        try:
            public_url = ngrok.connect(8501)
            print(f"🚀 Dashboard của bạn đã sẵn sàng! Truy cập tại đây: {public_url}")
        except Exception as e:
            print(f"LỖI KẾT NỐI NGROK: Đã xảy ra lỗi khi tạo tunnel. Chi tiết: {e}")
            print("Gợi ý: Hãy thử chạy lại cell này. Nếu vẫn lỗi, hãy kiểm tra lại Authtoken.")

Đang khởi động ứng dụng Streamlit trong nền...
Đang chờ ứng dụng khởi động (5 giây)...
🚀 Dashboard của bạn đã sẵn sàng! Truy cập tại đây: NgrokTunnel: "https://0a26d0c235cd.ngrok-free.app" -> "http://localhost:8501"
